In [28]:
# Preprocess Data for ML model
# Evaluating Fairness in Data-Driven Heat Vulnerability Risk Prediction
# 6.C511
# Justin Liaw, Shreeya Parekh, Julia Lukens

### Inputs:
    # HHI dataset (ZIP code level) — 25 indicator columns + Overall HHI Ranking column
    # CDC WONDER mortality data (county level) — heat-related death counts 
    # Census Bureau ZIP-to-county FIPS crosswalk

### Outputs:
# One merged dataframe at county level containing:
    # FIPS code
    # 25 population-weighted HHI indicator columns
    # Overall HHI Ranking column (population-weighted, aggregated to county)
    # Binary label column (1 = high-risk, 0 = low-risk)
    # Income quartile column (1-4)
    # Data quality column (number of ZIP codes averaged per county) (loawer priority)
    # Suppression flag column (boolean) - for counties where actual number of heat-related deaths was below 10 and CDC chose not to report it to protect privacy.

### Tasks:
    # Download HHI dataset and CDC WONDER mortality data
    # Download ZIP-to-FIPS crosswalk from Census Bureau
    # Merge crosswalk with HHI data to assign each ZIP code to a county
    # Aggregate all 25 HHI indicators and Overall HHI Ranking to county level using population-weighted averaging
    # Calculate heat-related mortality rate per county from CDC WONDER
    # Derive binary label — top quartile of mortality rate = 1, all others = 0
    # Derive income quartile column from the relevant HHI sociodemographic indicator
    # Flag counties with suppressed CDC WONDER counts
    # Add data quality column counting ZIP codes per county
    # Deliver clean merged dataframe with agreed-upon column names

In [29]:
import os
import pandas as pd
import numpy as np 
import glob
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [30]:
# Read in data

hhi = pd.read_excel("./HHI_data/HHI Data 2024 United States.xlsx", dtype={"ZCTA": str})

mortality = pd.read_csv(
    "./CDC_WONDER/Multiple Cause of Death, 1999-2020 (7).csv",
    dtype={
        "County Code": str
    }
)

xwalk = pd.read_csv(
    "./spatial_crosswalk/tab20_zcta520_county20_natl.txt", 
    sep="|",
    dtype={
        "GEOID_ZCTA5_20": str,
        "GEOID_COUNTY_20": str
    }
)

In [31]:
# Population-weighted aggregation of HHI data to county level and derivation of income quartile

# ----------------------------
# 1. Clean Columns
# ----------------------------
hhi["ZCTA"] = hhi["ZCTA"].astype(str).str.zfill(5)
xwalk["ZCTA"] = xwalk["GEOID_ZCTA5_20"].astype(str).str.zfill(5)
xwalk["county_fips"] = xwalk["GEOID_COUNTY_20"].astype(str).str.zfill(5)

# ----------------------------
# 2. Merge county crosswalk onto HHI ZCTAs
# ----------------------------
merged = hhi.merge(
    xwalk[["ZCTA", "county_fips", "NAMELSAD_COUNTY_20", "AREALAND_PART"]],
    on="ZCTA",
    how="inner"
)

# ----------------------------
# 3. Create ZCTA-to-county population weights
# ----------------------------
# We want to use population weighting to aggregate to county level.
# First, how much of each ZCTA’s population belongs to each county?
# The crosswalk does not have population,
# so approximate county-share population using land-area share.
# Assumption: population is evenly distributed within a ZCTA.
# Then use HHI POP to weight county aggregation.

# compute total area of each ZCTA
merged["zcta_area_total"] = merged.groupby("ZCTA")["AREALAND_PART"].transform("sum")  # AREALAND_PART = Calculated land area of the overlapping part in square meters

# compute area share = area in county X / total ZCTA area
merged["zcta_county_area_share"] = (merged["AREALAND_PART"] / merged["zcta_area_total"])

# allocate total ZCTA population across counties
merged["pop_allocated_to_county"] = (merged["POP"] * merged["zcta_county_area_share"])

# ----------------------------
# 4. Choose HHI columns to aggregate
# ----------------------------
# These are the HHI indicator/rank/score columns.
# Exclude IDs, names, and raw population (we'll use "pop_allocated_to_county" instead!).
exclude_cols = [
    "STATEFP10", "STATE", "STATE_ABV", "ZCTA", "GEOID10",
    "MULTI_STATE", "POP"
]

numeric_cols = hhi.select_dtypes(include=[np.number]).columns.tolist()

hhi_feature_cols = [
    c for c in numeric_cols
    if c not in exclude_cols
]

# ----------------------------
# 5. Population-weighted county averages
# ----------------------------

# fcn to calculate population-weighted average HHI indicator value for each county:
    # 1. group the merged ZCTA-county data by county
    # 2. for each county, compute population-weighted average of every HHI indicator using allocated ZCTA population as weights

def weighted_average(group, cols, weight_col):
    weights = group[weight_col] # "pop_allocated_to_county"
    out = {}

    for col in cols:
        values = group[col]
        mask = values.notna() & weights.notna() # only keep rows where both the indicator value and population weight exist

        if mask.sum() == 0 or weights[mask].sum() == 0:
            out[col] = np.nan
        else:
            out[col] = np.average(values[mask], weights=weights[mask]) # compute population-weighted average

    return pd.Series(out)

# apply fcn to each county
county_hhi = (
    merged
    .groupby(["county_fips", "NAMELSAD_COUNTY_20"])
    .apply(
        weighted_average,
        cols=hhi_feature_cols,
        weight_col="pop_allocated_to_county"
    )
    .reset_index()
)

# ----------------------------
# 6. Add total allocated population per county from all contributing ZCTAs
# ----------------------------
county_pop = (
    merged
    .groupby(["county_fips", "NAMELSAD_COUNTY_20"])["pop_allocated_to_county"]
    .sum()
    .reset_index()
    .rename(columns={"pop_allocated_to_county": "county_pop_from_zctas"})
)

county_hhi = county_hhi.merge(
    county_pop,
    on=["county_fips", "NAMELSAD_COUNTY_20"],
    how="left"
)

# ----------------------------
# 7. Add a data quality column counting ZIP codes per county (overlap)
# ----------------------------
county_zip_counts = (
    merged
    .groupby(["county_fips", "NAMELSAD_COUNTY_20"])["ZCTA"]
    .nunique()
    .reset_index()
    .rename(columns={"ZCTA": "n_zctas_in_county"})
)

county_hhi = county_hhi.merge(
    county_zip_counts,
    on=["county_fips", "NAMELSAD_COUNTY_20"],
    how="left"
)

# ----------------------------
# 8. Derive poverty quartile
# ----------------------------
# 1 = bottom 25%
# 4 = top 25%

# TODO recommend: 
# quartiles → exploratory analysis
# binary high_poverty → Fairlearn metrics
# because many fairness metrics work more cleanly with two groups

county_hhi["poverty_quartile"] = pd.qcut(
    county_hhi["PR_POV"],
    q=4,
    labels=[1, 2, 3, 4]
)

county_hhi["poverty_quartile"] = (
    county_hhi["poverty_quartile"]
    .astype(int)
)

# ----------------------------
# 9. Save output
# ----------------------------
county_hhi.to_csv("county_level_hhi_population_weighted.csv", index=False)

print(county_hhi.shape)
print(county_hhi.head())



/tmp/ipykernel_133263/1819722229.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


(3108, 73)
  county_fips NAMELSAD_COUNTY_20     PR_HRI     F_HRI   LOW_EMS      P_NEHD  \
0       01001     Autauga County  64.365804  0.756453  0.000000  -18.438789   
1       01003     Baldwin County  75.842151  0.297279  0.002448  -29.605982   
2       01005     Barbour County  56.780091  0.722064  0.001760    9.767859   
3       01007        Bibb County  72.725062  0.839975  0.004131 -216.191705   
4       01009      Blount County  59.495221 -1.335081 -2.047257  -57.887124   

      PR_NEHD  HHB_SCORE  HHB_RANK     P_CHD  ...  PR_OZONE    P_PM25  \
0  -26.215642   0.479537  0.372070  5.978737  ...  0.377930  0.000000   
1  -38.983087   0.595870  0.566914  6.719081  ...  0.217727  0.000000   
2    0.270125   0.496047  0.390836  8.424020  ...  0.000000  0.000000   
3 -221.100070   0.559280  0.497766  6.775043  ...  0.528481  0.000000   
4  -61.469182  -1.639118 -1.791044  7.291256  ...  0.295537  0.004112   

    PR_PM25  NBE_SCORE  NBE_RANK  OVERALL_SCORE  OVERALL_RANK  \
0  0.00000

In [33]:
# Shannon County SD (old FIPS 46113) was renamed Oglala Lakota County (new FIPS 46102) in 2015.
# HHI uses the new code, CDC WONDER mortality data uses the old code.
# Remap mortality to match HHI before merging.
# Create county_fips from County Code and remap Oglala Lakota in one step
mortality["county_fips"] = (
    mortality["County Code"]
    .astype(str)
    .str.zfill(5)
    .replace("46113", "46102")
)

In [34]:
# MICE imputation for suppressed counties
mortality["Deaths_adjusted"] = (
    mortality["Deaths"]
    .replace("Suppressed", np.nan)
    .astype(float)
)
mortality["suppression_flag"] = mortality["Deaths"].eq("Suppressed").astype(int)

mice_df = mortality[["Deaths_adjusted", "Population"]].copy()
imputer = IterativeImputer(
    max_iter=10,
    random_state=42,
    min_value=1,
    max_value=9,
    sample_posterior=True
)
imputed_array = imputer.fit_transform(mice_df)
mortality["Deaths_adjusted"] = np.round(imputed_array[:, 0]).astype(int)

# Crude rate per 100,000 population — standard epidemiological convention
mortality["Crude Rate_adjusted"] = (
    mortality["Deaths_adjusted"] / mortality["Population"]
) * 100_000

# Define masks
observed_mask         = mortality["suppression_flag"] == 0
observed_nonzero_mask = observed_mask & (mortality["Crude Rate_adjusted"] > 0)

# Compute threshold on observed non-zero counties only
# Prevents suppressed counties with small imputed counts and small
# populations from inflating crude rates into the top quartile
threshold = mortality.loc[observed_nonzero_mask, "Crude Rate_adjusted"].quantile(0.75)

print(f"Threshold computed on {observed_nonzero_mask.sum()} observed non-zero counties: {threshold:.6f}")
print(f"High-risk among observed  : {(mortality.loc[observed_mask,  'Crude Rate_adjusted'] >= threshold).sum()}")
print(f"High-risk among suppressed: {(mortality.loc[~observed_mask, 'Crude Rate_adjusted'] >= threshold).sum()}")

# Zero-death counties are explicitly low-risk regardless of threshold
mortality["label"] = 0
mortality.loc[mortality["Crude Rate_adjusted"] >= threshold, "label"] = 1

print(f"\nLabel distribution:")
print(mortality["label"].value_counts())
print(f"High-risk rate: {mortality['label'].mean():.1%}")

Threshold computed on 233 observed non-zero counties: 0.413051
High-risk among observed  : 59
High-risk among suppressed: 560

Label distribution:
label
0    2528
1     619
Name: count, dtype: int64
High-risk rate: 19.7%


In [35]:
persistent_poverty = pd.read_excel(
    "./Pers_Pov/Persistent-Poverty-County-List-Rural-Development.xlsx",
    dtype={"FIPS": str}
)
persistent_poverty["FIPS"] = persistent_poverty["FIPS"].str.zfill(5)

county_hhi["income_group"] = county_hhi["county_fips"].isin(
    persistent_poverty["FIPS"]
).astype(int)

n_persistent = county_hhi["income_group"].sum()
print(f"Persistent poverty counties    : {n_persistent}")
print(f"Non-persistent poverty counties: {(county_hhi['income_group']==0).sum()}")
print(f"Persistent poverty share       : {n_persistent/len(county_hhi):.1%}")

Persistent poverty counties    : 392
Non-persistent poverty counties: 2716
Persistent poverty share       : 12.6%


In [36]:
county_hhi = county_hhi.merge(
    mortality[["county_fips", "Deaths_adjusted", "Population",
               "Crude Rate_adjusted", "suppression_flag", "label"]],
    on="county_fips",
    how="left"
)

print(f"Merged shape: {county_hhi.shape}")
print(f"NaN labels  : {county_hhi['label'].isna().sum()}")
print(f"High-risk   : {county_hhi['label'].sum()} ({county_hhi['label'].mean():.1%})")

Merged shape: (3108, 79)
NaN labels  : 0
High-risk   : 613 (19.7%)


In [37]:
county_hhi.to_csv("./heat_risk_dataframe.csv", index=False)

# Validate
test = pd.read_csv("./heat_risk_dataframe.csv")
print("Columns present:")
for col in ["label", "suppression_flag", "income_group", "poverty_quartile", "OVERALL_RANK"]:
    print(f"  {col}: {'✓' if col in test.columns else '✗ MISSING'}")
print(f"\nLabel distribution: {test['label'].value_counts().to_dict()}")
print(f"income_group distribution: {test['income_group'].value_counts().to_dict()}")

Columns present:
  label: ✓
  suppression_flag: ✓
  income_group: ✓
  poverty_quartile: ✓
  OVERALL_RANK: ✓

Label distribution: {0: 2495, 1: 613}
income_group distribution: {0: 2716, 1: 392}
